# Berliner Straßen + Adressen — Export für StreetGuesser

Lädt Berliner Straßenabschnitte (RBS), führt sie nach Straßenname zusammen,
und bereitet den Datensatz für das StreetGuesser-Spiel vor.

**Ausgabe:**
- `processed_data/strassen_berlin.geojson` (EPSG:25833)
- `processed_data/strassen_berlin_wgs84.geojson` (EPSG:4326)

**Felder im Output:**
- `strname` — Straßenname
- `bezname` — Bezirk(e)
- `laenge` — Länge in Metern
- `n_bezirke` — Anzahl Bezirke
- `strnr` — Straßennummer (RBS-ID)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load raw Berlin street segments (WGS84), reproject to metric CRS
raw = gpd.read_file('../storytelling/data/strassen_berlin_wgs84.geojson')
raw_utm = raw.to_crs(epsg=25833)

print(f'{len(raw_utm):,} Straßensegmente')
print(f'CRS: {raw_utm.crs}')
print(f'Felder: {raw_utm.columns.tolist()}')

## Segmente nach Straßenname zusammenführen

In [ ]:
# Dissolve by street name — one geometry per unique strname
strassen = raw_utm.dissolve(by='strname', aggfunc={
    'strnr': 'first',
    'bezname': lambda x: ', '.join(sorted(set(x))),
    'strabtypname': 'first'
}).reset_index()

# Calculate length in metres
strassen['laenge'] = strassen.geometry.length.round(1)

# Count how many Bezirke each street crosses
strassen['n_bezirke'] = raw_utm.groupby('strname')['bezname'].nunique().reindex(strassen['strname']).values

print(f'{len(strassen):,} eindeutige Straßennamen')
strassen.sort_values('laenge', ascending=False).head(10)[['strname', 'laenge', 'n_bezirke', 'bezname']]

## Qualitätskontrolle

In [ ]:
print('=== Fehlende Straßennamen ===')
print(strassen['strname'].isna().sum())

print('\n=== Straßen ohne Geometrie ===')
print(strassen[strassen.is_empty].shape[0])

print('\n=== Längenverteilung ===')
print(strassen['laenge'].describe().round(1))

## Längste und kürzeste Straßen

In [ ]:
EXCLUDE = ['BAB', 'Wasserstraße', 'kanal', 'Kanal', 'Havel', 'Spree', 'Wuhle',
           'Panke', 'Dahme', 'Nordgraben', 'Teltow', 'BLW', 'Verbindungsweg']

proper = strassen[~strassen['strname'].str.contains('|'.join(EXCLUDE), na=False)].copy()

print('=== TOP 10 längste Straßen (ohne Autobahnen/Wasserstraßen) ===')
top10 = proper.nlargest(10, 'laenge')[['strname', 'laenge', 'n_bezirke', 'bezname']].copy()
top10['laenge_km'] = (top10['laenge'] / 1000).round(2)
display(top10[['strname', 'laenge_km', 'n_bezirke', 'bezname']])

print('\n=== Kürzeste Straßen ===')
bottom10 = proper.nsmallest(10, 'laenge')[['strname', 'laenge', 'bezname']].copy()
bottom10['laenge_m'] = bottom10['laenge'].round(1)
display(bottom10[['strname', 'laenge_m', 'bezname']])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top10_plot = proper.nlargest(10, 'laenge').copy()
top10_plot['laenge_km'] = top10_plot['laenge'] / 1000
ax.barh(top10_plot['strname'], top10_plot['laenge_km'], color='#e74c3c')
ax.set_xlabel('Länge (km)')
ax.set_title('Top 10 längste Straßen Berlins')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Bezirk-Statistiken

In [ ]:
bez_stats = raw_utm.groupby('bezname').agg(
    n_segmente=('gisid', 'count'),
    n_strassen=('strname', 'nunique'),
    gesamt_laenge_km=('geometry', lambda x: round(x.length.sum() / 1000, 1))
).sort_values('n_strassen', ascending=False)

print(bez_stats)

fig, ax = plt.subplots(figsize=(10, 5))
bez_stats['n_strassen'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Eindeutige Straßennamen pro Bezirk')
ax.set_xlabel('Anzahl')
plt.tight_layout()
plt.show()

## Export für StreetGuesser

In [ ]:
# Select and reorder output columns
output_cols = ['strname', 'strnr', 'bezname', 'n_bezirke', 'laenge', 'strabtypname', 'geometry']
export = strassen[output_cols].copy()

out_dir = Path('../processed_data')
out_dir.mkdir(exist_ok=True)

# Export EPSG:25833 (native metric CRS for Berlin)
export.to_file(out_dir / 'strassen_berlin.geojson', driver='GeoJSON')
print(f'Saved strassen_berlin.geojson ({len(export):,} Straßen)')

# Export WGS84 for web use
export.to_crs(epsg=4326).to_file(out_dir / 'strassen_berlin_wgs84.geojson', driver='GeoJSON')
print(f'Saved strassen_berlin_wgs84.geojson')

## Karte: Straßennetz nach Bezirk

In [ ]:
# Color by Bezirk — requires loading bezirk boundaries for the background
fig, ax = plt.subplots(figsize=(12, 12), facecolor='#1a1a1a')
raw_utm.plot(ax=ax, column='bezname', linewidth=0.15, alpha=0.6, cmap='tab20', legend=False)
ax.set_title('Berliner Straßennetz nach Bezirk', color='white', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()